# FDIC BankFind → Databricks Bronze Ingestion

Ingests three FDIC BankFind API datasets into Delta bronze tables:
1. **Institution master** — `bronze.fdic_institutions`
2. **Quarterly financials** — `bronze.fdic_financials_quarterly`
3. **Bank failures** — `bronze.fdic_failures`

No API key required. Supports BACKFILL (from 1992) and INCREMENTAL load modes.

In [0]:
# ============================================================
# 1. IMPORTS
# ============================================================

import time
import requests

from datetime import date, datetime

from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    DoubleType,
    DateType,
    TimestampType
)

from delta.tables import DeltaTable


spark = SparkSession.builder.getOrCreate()

In [0]:
# ============================================================
# 2. CONFIGURATION
# ============================================================

BASE_URL = "https://api.fdic.gov/banks"

INSTITUTIONS_ENDPOINT = f"{BASE_URL}/institutions"
FINANCIALS_ENDPOINT = f"{BASE_URL}/financials"
FAILURES_ENDPOINT = f"{BASE_URL}/failures"


INSTITUTIONS_TABLE = "bronze.fdic_institutions"

FINANCIALS_TABLE = "bronze.fdic_financials_quarterly"

FAILURES_TABLE = "bronze.fdic_failures"


# ------------------------------------------------------------
# LOAD MODE
# ------------------------------------------------------------
#
# BACKFILL:
#   Pull quarterly financials beginning in 1992.
#
# INCREMENTAL:
#   Look at the latest reporting period already stored and
#   re-fetch that quarter plus any newer quarter.
#
# Re-fetching the latest quarter is intentional because
# institutions can report/update at different times.
# ------------------------------------------------------------

LOAD_MODE = "BACKFILL"


# FDIC quarterly financial reporting is available back to 1992.

BACKFILL_START_DATE = date(1992, 3, 31)


# ------------------------------------------------------------
# DATASETS TO INGEST
# ------------------------------------------------------------

INGEST_INSTITUTIONS = True
INGEST_FINANCIALS = True
INGEST_FAILURES = True


# ------------------------------------------------------------
# REQUEST SETTINGS
# ------------------------------------------------------------

PAGE_SIZE = 10000

REQUEST_TIMEOUT = 60

MAX_RETRIES = 4

RETRY_DELAY_SECONDS = 5

# Be polite to the public API.
REQUEST_DELAY_SECONDS = 0.25


HEADERS = {
    "Accept": "application/json",
    "User-Agent": "Databricks-FDIC-Research-Pipeline"
}

In [0]:
# ============================================================
# 3. CREATE BRONZE SCHEMA
# ============================================================

spark.sql("""
CREATE SCHEMA IF NOT EXISTS bronze
""")

In [0]:
# ============================================================
# 4. HELPER FUNCTIONS
# ============================================================


def safe_int(value):
    """
    Convert FDIC values into Python integers.
    Returns None for blank or malformed values.
    """

    if value is None:
        return None

    value = str(value).strip().replace(",", "")

    if value in ("", "None", "null", "NULL"):
        return None

    try:
        return int(float(value))

    except (ValueError, TypeError):
        return None



def safe_float(value):
    """
    Convert FDIC numeric values into floats.
    """

    if value is None:
        return None

    value = str(value).strip().replace(",", "")

    if value in ("", "None", "null", "NULL"):
        return None

    try:
        return float(value)

    except (ValueError, TypeError):
        return None



def safe_date(value):
    """
    Parse common FDIC date formats.
    """

    if value is None:
        return None

    value = str(value).strip()

    if not value:
        return None

    formats = [
        "%Y-%m-%d",
        "%Y%m%d",
        "%m/%d/%Y"
    ]

    for fmt in formats:

        try:
            return datetime.strptime(
                value,
                fmt
            ).date()

        except ValueError:
            pass

    return None



def unwrap_fdic_record(record):
    """
    FDIC responses may wrap fields inside a `data`
    object. This handles both wrapped and flat forms.
    """

    if (
        isinstance(record, dict)
        and isinstance(record.get("data"), dict)
    ):
        return record["data"]

    return record



def fdic_get_all(
    endpoint,
    fields,
    filters=None,
    sort_by=None,
    sort_order="ASC"
):
    """
    Retrieve all records for a FDIC API request using
    limit/offset pagination.
    """

    all_records = []

    offset = 0


    while True:

        params = {
            "fields": ",".join(fields),
            "limit": PAGE_SIZE,
            "offset": offset,
            "format": "json"
        }


        if filters:
            params["filters"] = filters


        if sort_by:
            params["sort_by"] = sort_by
            params["sort_order"] = sort_order


        response = None


        for attempt in range(
            1,
            MAX_RETRIES + 1
        ):

            try:

                response = requests.get(
                    endpoint,
                    params=params,
                    headers=HEADERS,
                    timeout=REQUEST_TIMEOUT
                )


                # --------------------------------------------
                # RETRY RATE LIMIT / SERVER ERRORS
                # --------------------------------------------

                if (
                    response.status_code == 429
                    or response.status_code >= 500
                ):

                    print(
                        f"HTTP {response.status_code}. "
                        f"Retry {attempt}/{MAX_RETRIES}"
                    )

                    time.sleep(
                        RETRY_DELAY_SECONDS * attempt
                    )

                    continue


                response.raise_for_status()

                break


            except requests.exceptions.RequestException:

                if attempt == MAX_RETRIES:
                    raise

                time.sleep(
                    RETRY_DELAY_SECONDS * attempt
                )


        payload = response.json()


        raw_records = payload.get(
            "data",
            []
        )


        records = [
            unwrap_fdic_record(record)
            for record in raw_records
        ]


        all_records.extend(records)


        page_count = len(records)


        total = (
            payload
            .get("meta", {})
            .get("total")
        )


        if total is not None:

            try:
                total = int(total)

            except (ValueError, TypeError):
                total = None


        print(
            f"Retrieved {len(all_records):,}"
            + (
                f" of {total:,}"
                if total is not None
                else ""
            )
        )


        # --------------------------------------------
        # EXIT CONDITIONS
        # --------------------------------------------

        if page_count == 0:
            break


        if (
            total is not None
            and len(all_records) >= total
        ):
            break


        if page_count < PAGE_SIZE:
            break


        offset += PAGE_SIZE

        time.sleep(
            REQUEST_DELAY_SECONDS
        )


    return all_records



def merge_delta(
    dataframe,
    target_table,
    merge_condition
):
    """
    Generic idempotent Delta MERGE.
    """

    target = DeltaTable.forName(
        spark,
        target_table
    )


    (
        target.alias("target")

        .merge(
            dataframe.alias("source"),
            merge_condition
        )

        .whenMatchedUpdateAll()

        .whenNotMatchedInsertAll()

        .execute()
    )

In [0]:
# ============================================================
# 5. CREATE TARGET TABLES
# ============================================================


# ------------------------------------------------------------
# INSTITUTION MASTER
# ------------------------------------------------------------

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {INSTITUTIONS_TABLE} (

    cert BIGINT,

    bank_name STRING,

    city STRING,

    state STRING,

    active BIGINT,

    total_assets_thousands BIGINT,

    total_deposits_thousands BIGINT,

    established_date DATE,

    data_source STRING,

    ingestion_timestamp TIMESTAMP

)
USING DELTA

COMMENT 'FDIC BankFind institution master'
""")


# ------------------------------------------------------------
# QUARTERLY FINANCIALS
# ------------------------------------------------------------

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {FINANCIALS_TABLE} (

    cert BIGINT,

    bank_name STRING,

    report_date DATE,

    total_assets_thousands BIGINT,

    total_deposits_thousands BIGINT,

    equity_capital_thousands BIGINT,

    net_income_thousands BIGINT,

    net_loans_thousands BIGINT,

    real_estate_loans_thousands BIGINT,

    construction_land_dev_loans_thousands BIGINT,

    multifamily_re_loans_thousands BIGINT,

    nonfarm_nonres_re_loans_thousands BIGINT,

    residential_1_4_family_loans_thousands BIGINT,

    noncurrent_loans_thousands BIGINT,

    nonaccrual_loans_thousands BIGINT,

    past_due_90_plus_thousands BIGINT,

    net_chargeoffs_thousands BIGINT,

    roa_pct DOUBLE,

    roe_pct DOUBLE,

    net_interest_margin_pct DOUBLE,

    data_source STRING,

    source_unit STRING,

    ingestion_timestamp TIMESTAMP

)
USING DELTA

COMMENT 'FDIC quarterly bank financial and regulatory data'
""")


# ------------------------------------------------------------
# FAILURE HISTORY
# ------------------------------------------------------------

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {FAILURES_TABLE} (

    failure_id STRING,

    cert BIGINT,

    bank_name STRING,

    city STRING,

    state STRING,

    failure_date DATE,

    failure_year BIGINT,

    assets_at_failure_thousands BIGINT,

    deposits_at_failure_thousands BIGINT,

    uninsured_deposits_thousands BIGINT,

    estimated_cost_thousands BIGINT,

    resolution_type STRING,

    acquiring_bank STRING,

    acquiring_city STRING,

    acquiring_state STRING,

    data_source STRING,

    source_unit STRING,

    ingestion_timestamp TIMESTAMP

)
USING DELTA

COMMENT 'FDIC historical bank failures and resolution information'
""")

In [0]:
# ============================================================
# 6. INGEST INSTITUTION MASTER
# ============================================================

if INGEST_INSTITUTIONS:

    print("\n========================================")
    print("INGESTING FDIC INSTITUTIONS")
    print("========================================")


    institution_fields = [

        "CERT",
        "NAME",
        "CITY",
        "STALP",
        "ACTIVE",
        "ASSET",
        "DEP",
        "ESTYMD"

    ]


    records = fdic_get_all(

        endpoint=INSTITUTIONS_ENDPOINT,

        fields=institution_fields,

        sort_by="CERT",

        sort_order="ASC"

    )


    institution_rows = []


    for record in records:

        institution_rows.append({

            "cert":
                safe_int(record.get("CERT")),

            "bank_name":
                record.get("NAME"),

            "city":
                record.get("CITY"),

            "state":
                record.get("STALP"),

            "active":
                safe_int(record.get("ACTIVE")),

            "total_assets_thousands":
                safe_int(record.get("ASSET")),

            "total_deposits_thousands":
                safe_int(record.get("DEP")),

            "established_date":
                safe_date(record.get("ESTYMD")),

            "data_source":
                "FDIC BankFind"

        })


    institution_schema = StructType([

        StructField(
            "cert",
            LongType(),
            True
        ),

        StructField(
            "bank_name",
            StringType(),
            True
        ),

        StructField(
            "city",
            StringType(),
            True
        ),

        StructField(
            "state",
            StringType(),
            True
        ),

        StructField(
            "active",
            LongType(),
            True
        ),

        StructField(
            "total_assets_thousands",
            LongType(),
            True
        ),

        StructField(
            "total_deposits_thousands",
            LongType(),
            True
        ),

        StructField(
            "established_date",
            DateType(),
            True
        ),

        StructField(
            "data_source",
            StringType(),
            False
        )

    ])


    institution_df = spark.createDataFrame(

        institution_rows,
        schema=institution_schema

    ).withColumn(

        "ingestion_timestamp",

        current_timestamp()

    )


    institution_df = institution_df.dropDuplicates([
        "cert"
    ])


    merge_delta(

        institution_df,

        INSTITUTIONS_TABLE,

        """
        target.cert = source.cert
        """

    )


    print(
        f"Institutions merged: "
        f"{institution_df.count():,}"
    )

In [0]:
# ============================================================
# 7. QUARTER-END DATE GENERATOR
# ============================================================


def quarter_end_dates(
    start_date,
    end_date
):

    quarter_dates = []


    for year in range(
        start_date.year,
        end_date.year + 1
    ):

        dates = [

            date(year, 3, 31),

            date(year, 6, 30),

            date(year, 9, 30),

            date(year, 12, 31)

        ]


        for quarter_date in dates:

            if (
                quarter_date >= start_date
                and quarter_date <= end_date
            ):

                quarter_dates.append(
                    quarter_date
                )


    return quarter_dates


# ============================================================
# 8. DETERMINE FINANCIAL START DATE
# ============================================================

financial_start_date = BACKFILL_START_DATE


if LOAD_MODE == "INCREMENTAL":

    existing_max = spark.sql(f"""

        SELECT MAX(report_date)
            AS max_report_date

        FROM {FINANCIALS_TABLE}

    """).collect()[0]["max_report_date"]


    if existing_max is not None:

        # Re-load the most recent quarter.
        # This catches late / revised reporting.

        financial_start_date = existing_max


elif LOAD_MODE != "BACKFILL":

    raise ValueError(
        "LOAD_MODE must be BACKFILL or INCREMENTAL"
    )


quarter_dates = quarter_end_dates(

    financial_start_date,

    date.today()

)


print(
    f"\nFinancial quarters to process: "
    f"{len(quarter_dates)}"
)

In [0]:
# ============================================================
# 9. FINANCIAL FIELD LIST
# ============================================================
#
# Important FDIC source fields:
#
# ASSET    Total assets
# DEP      Total deposits
# EQ       Equity capital
# NETINC   Net income
# LNLSNET  Net loans and leases
#
# LNRE     Real-estate loans
# LNRECONS Construction & land development
# LNREMULT Multifamily
# LNRENRES Nonfarm nonresidential
# LNRERES  1-4 family residential
#
# NCLNLS   90+ days past due + nonaccrual
# NALNLS   Nonaccrual loans
# P9LNLS   90+ days past due
# NCHGREC  Net charge-offs
#
# ROA      Return on assets
# ROE      Return on equity
# NIM      Net interest margin
#
# Monetary values are generally reported in thousands of USD.
# ============================================================


financial_fields = [

    "CERT",
    "NAME",
    "REPDTE",

    "ASSET",
    "DEP",
    "EQ",
    "NETINC",

    "LNLSNET",

    "LNRE",
    "LNRECONS",
    "LNREMULT",
    "LNRENRES",
    "LNRERES",

    "NCLNLS",
    "NALNLS",
    "P9LNLS",
    "NCHGREC",

    "ROA",
    "ROE",
    "NIM"

]


# ============================================================
# 10. FINANCIAL SPARK SCHEMA
# ============================================================

financial_schema = StructType([

    StructField(
        "cert",
        LongType(),
        False
    ),

    StructField(
        "bank_name",
        StringType(),
        True
    ),

    StructField(
        "report_date",
        DateType(),
        False
    ),

    StructField(
        "total_assets_thousands",
        LongType(),
        True
    ),

    StructField(
        "total_deposits_thousands",
        LongType(),
        True
    ),

    StructField(
        "equity_capital_thousands",
        LongType(),
        True
    ),

    StructField(
        "net_income_thousands",
        LongType(),
        True
    ),

    StructField(
        "net_loans_thousands",
        LongType(),
        True
    ),

    StructField(
        "real_estate_loans_thousands",
        LongType(),
        True
    ),

    StructField(
        "construction_land_dev_loans_thousands",
        LongType(),
        True
    ),

    StructField(
        "multifamily_re_loans_thousands",
        LongType(),
        True
    ),

    StructField(
        "nonfarm_nonres_re_loans_thousands",
        LongType(),
        True
    ),

    StructField(
        "residential_1_4_family_loans_thousands",
        LongType(),
        True
    ),

    StructField(
        "noncurrent_loans_thousands",
        LongType(),
        True
    ),

    StructField(
        "nonaccrual_loans_thousands",
        LongType(),
        True
    ),

    StructField(
        "past_due_90_plus_thousands",
        LongType(),
        True
    ),

    StructField(
        "net_chargeoffs_thousands",
        LongType(),
        True
    ),

    StructField(
        "roa_pct",
        DoubleType(),
        True
    ),

    StructField(
        "roe_pct",
        DoubleType(),
        True
    ),

    StructField(
        "net_interest_margin_pct",
        DoubleType(),
        True
    ),

    StructField(
        "data_source",
        StringType(),
        False
    ),

    StructField(
        "source_unit",
        StringType(),
        False
    )

])

In [0]:
# ============================================================
# 11. INGEST QUARTERLY FINANCIALS
# ============================================================
#
# Process quarter by quarter.
#
# This avoids loading the entire 1992-present history into
# Python memory before writing to Delta.
# ============================================================

if INGEST_FINANCIALS:

    print("\n========================================")
    print("INGESTING FDIC QUARTERLY FINANCIALS")
    print("========================================")


    total_financial_rows = 0


    for index, quarter_date in enumerate(
        quarter_dates,
        start=1
    ):

        quarter_string = quarter_date.strftime(
            "%Y-%m-%d"
        )


        print(
            f"\n[{index}/{len(quarter_dates)}] "
            f"Reporting period: {quarter_string}"
        )


        records = fdic_get_all(

            endpoint=FINANCIALS_ENDPOINT,

            fields=financial_fields,

            filters=(
                f"REPDTE:{quarter_string}"
            ),

            sort_by="CERT",

            sort_order="ASC"

        )


        if not records:

            print(
                f"No records returned for "
                f"{quarter_string}"
            )

            continue


        rows = []


        for record in records:

            cert = safe_int(
                record.get("CERT")
            )

            report_date = safe_date(
                record.get("REPDTE")
            )


            if (
                cert is None
                or report_date is None
            ):
                continue


            rows.append({

                "cert":
                    cert,

                "bank_name":
                    record.get("NAME"),

                "report_date":
                    report_date,

                "total_assets_thousands":
                    safe_int(
                        record.get("ASSET")
                    ),

                "total_deposits_thousands":
                    safe_int(
                        record.get("DEP")
                    ),

                "equity_capital_thousands":
                    safe_int(
                        record.get("EQ")
                    ),

                "net_income_thousands":
                    safe_int(
                        record.get("NETINC")
                    ),

                "net_loans_thousands":
                    safe_int(
                        record.get("LNLSNET")
                    ),

                "real_estate_loans_thousands":
                    safe_int(
                        record.get("LNRE")
                    ),

                "construction_land_dev_loans_thousands":
                    safe_int(
                        record.get("LNRECONS")
                    ),

                "multifamily_re_loans_thousands":
                    safe_int(
                        record.get("LNREMULT")
                    ),

                "nonfarm_nonres_re_loans_thousands":
                    safe_int(
                        record.get("LNRENRES")
                    ),

                "residential_1_4_family_loans_thousands":
                    safe_int(
                        record.get("LNRERES")
                    ),

                "noncurrent_loans_thousands":
                    safe_int(
                        record.get("NCLNLS")
                    ),

                "nonaccrual_loans_thousands":
                    safe_int(
                        record.get("NALNLS")
                    ),

                "past_due_90_plus_thousands":
                    safe_int(
                        record.get("P9LNLS")
                    ),

                "net_chargeoffs_thousands":
                    safe_int(
                        record.get("NCHGREC")
                    ),

                "roa_pct":
                    safe_float(
                        record.get("ROA")
                    ),

                "roe_pct":
                    safe_float(
                        record.get("ROE")
                    ),

                "net_interest_margin_pct":
                    safe_float(
                        record.get("NIM")
                    ),

                "data_source":
                    "FDIC BankFind",

                "source_unit":
                    "USD thousands"

            })


        if not rows:

            continue


        quarter_df = (

            spark.createDataFrame(
                rows,
                schema=financial_schema
            )

            .withColumn(
                "ingestion_timestamp",
                current_timestamp()
            )

            .dropDuplicates([
                "cert",
                "report_date"
            ])

        )


        merge_delta(

            quarter_df,

            FINANCIALS_TABLE,

            """

            target.cert = source.cert

            AND

            target.report_date =
                source.report_date

            """

        )


        quarter_count = quarter_df.count()
        total_financial_rows += quarter_count


        print(
            f"Merged {quarter_count:,} "
            f"records for {quarter_string}"
        )


    print(
        f"\nTotal financial observations processed: "
        f"{total_financial_rows:,}"
    )

In [0]:
# ============================================================
# 12. INGEST BANK FAILURES
# ============================================================

if INGEST_FAILURES:

    print("\n========================================")
    print("INGESTING FDIC FAILURE HISTORY")
    print("========================================")


    failure_fields = [

        "ID",
        "CERT",
        "NAME",

        "CITY",
        "PSTALP",

        "FAILDATE",
        "FAILYR",

        "QBFASSET",
        "QBFDEP",
        "UNINSDEP",
        "COST",

        "RESTYPE",

        "BIDNAME",
        "BIDCITY",
        "BIDSTATE"

    ]


    records = fdic_get_all(

        endpoint=FAILURES_ENDPOINT,

        fields=failure_fields,

        sort_by="FAILDATE",

        sort_order="ASC"

    )


    failure_rows = []


    for record in records:

        failure_id = record.get("ID")


        if failure_id is None:

            # Fallback identifier if needed.

            failure_id = (
                f"{record.get('CERT')}_"
                f"{record.get('FAILDATE')}"
            )


        failure_rows.append({

            "failure_id":
                str(failure_id),

            "cert":
                safe_int(
                    record.get("CERT")
                ),

            "bank_name":
                record.get("NAME"),

            "city":
                record.get("CITY"),

            "state":
                record.get("PSTALP"),

            "failure_date":
                safe_date(
                    record.get("FAILDATE")
                ),

            "failure_year":
                safe_int(
                    record.get("FAILYR")
                ),

            "assets_at_failure_thousands":
                safe_int(
                    record.get("QBFASSET")
                ),

            "deposits_at_failure_thousands":
                safe_int(
                    record.get("QBFDEP")
                ),

            "uninsured_deposits_thousands":
                safe_int(
                    record.get("UNINSDEP")
                ),

            "estimated_cost_thousands":
                safe_int(
                    record.get("COST")
                ),

            "resolution_type":
                record.get("RESTYPE"),

            "acquiring_bank":
                record.get("BIDNAME"),

            "acquiring_city":
                record.get("BIDCITY"),

            "acquiring_state":
                record.get("BIDSTATE"),

            "data_source":
                "FDIC BankFind",

            "source_unit":
                "USD thousands"

        })


    failure_schema = StructType([

        StructField(
            "failure_id",
            StringType(),
            False
        ),

        StructField(
            "cert",
            LongType(),
            True
        ),

        StructField(
            "bank_name",
            StringType(),
            True
        ),

        StructField(
            "city",
            StringType(),
            True
        ),

        StructField(
            "state",
            StringType(),
            True
        ),

        StructField(
            "failure_date",
            DateType(),
            True
        ),

        StructField(
            "failure_year",
            LongType(),
            True
        ),

        StructField(
            "assets_at_failure_thousands",
            LongType(),
            True
        ),

        StructField(
            "deposits_at_failure_thousands",
            LongType(),
            True
        ),

        StructField(
            "uninsured_deposits_thousands",
            LongType(),
            True
        ),

        StructField(
            "estimated_cost_thousands",
            LongType(),
            True
        ),

        StructField(
            "resolution_type",
            StringType(),
            True
        ),

        StructField(
            "acquiring_bank",
            StringType(),
            True
        ),

        StructField(
            "acquiring_city",
            StringType(),
            True
        ),

        StructField(
            "acquiring_state",
            StringType(),
            True
        ),

        StructField(
            "data_source",
            StringType(),
            False
        ),

        StructField(
            "source_unit",
            StringType(),
            False
        )

    ])


    failure_df = (

        spark.createDataFrame(
            failure_rows,
            schema=failure_schema
        )

        .withColumn(
            "ingestion_timestamp",
            current_timestamp()
        )

        .dropDuplicates([
            "failure_id"
        ])

    )


    merge_delta(

        failure_df,

        FAILURES_TABLE,

        """
        target.failure_id =
            source.failure_id
        """

    )


    print(
        f"Failure records merged: "
        f"{failure_df.count():,}"
    )

In [0]:
# ============================================================
# 13. VALIDATION
# ============================================================

print("\n========================================")
print("FDIC INGESTION VALIDATION")
print("========================================")


if INGEST_INSTITUTIONS:

    display(
        spark.sql(f"""

        SELECT

            COUNT(*) AS institution_records,

            COUNT(DISTINCT cert)
                AS unique_banks,

            SUM(
                CASE
                    WHEN active = 1
                    THEN 1
                    ELSE 0
                END
            ) AS active_banks

        FROM {INSTITUTIONS_TABLE}

        """)
    )


if INGEST_FINANCIALS:

    display(
        spark.sql(f"""

        SELECT

            COUNT(*) AS financial_records,

            COUNT(DISTINCT cert)
                AS unique_banks,

            MIN(report_date)
                AS earliest_period,

            MAX(report_date)
                AS latest_period

        FROM {FINANCIALS_TABLE}

        """)
    )


if INGEST_FAILURES:

    display(
        spark.sql(f"""

        SELECT

            COUNT(*) AS failure_records,

            MIN(failure_date)
                AS earliest_failure,

            MAX(failure_date)
                AS latest_failure

        FROM {FAILURES_TABLE}

        """)
    )


# ============================================================
# 14. SAMPLE FINANCIAL DATA
# ============================================================

if INGEST_FINANCIALS:

    display(
        spark.sql(f"""

        SELECT

            cert,

            bank_name,

            report_date,

            total_assets_thousands,

            total_deposits_thousands,

            equity_capital_thousands,

            net_loans_thousands,

            noncurrent_loans_thousands,

            roa_pct,

            roe_pct,

            net_interest_margin_pct

        FROM {FINANCIALS_TABLE}

        ORDER BY

            report_date DESC,

            total_assets_thousands DESC

        LIMIT 100

        """)
    )


# ============================================================
# 15. INGESTION COMPLETE
# ============================================================

print("\n========================================")
print("FDIC INGESTION COMPLETE")
print("========================================")

print(
    f"Load mode: {LOAD_MODE}"
)

print(
    f"Institutions table: "
    f"{INSTITUTIONS_TABLE}"
)

print(
    f"Financials table: "
    f"{FINANCIALS_TABLE}"
)

print(
    f"Failures table: "
    f"{FAILURES_TABLE}"
)

print("========================================")